# Campaign Performance Classification & Acquisition Cost Analysis

Building on the RFM segmentation established in our clustering analysis, this notebook investigates **which campaigns convert which customer segments** and quantifies the **financial impact of targeted vs untargeted campaign spending**.

**Part 1 — Exploratory Analysis** (Sections 1–4):  
Campaign acceptance rates by segment, channel purchase preferences, product spend profiles, and a segment overview dashboard. Key finding: past campaign acceptance is the single strongest predictor of future response (40.7% vs 8.3%).

**Part 2 — Predictive Modeling** (Section 5):  
A Gradient Boosting classifier trained on customer × campaign pairs to predict acceptance probability. Feature importances confirm that Income, Recency, and Total Spending drive campaign responsiveness — validating our RFM segmentation approach.

**Part 3 — CAC & ROI Optimization** (Sections 6–7):  
Three targeting strategies compared on a $500 budget: spray-and-pray (S1), best campaign for everyone (S2), and best campaign targeted to past responders (S3). We compute Customer Acquisition Cost, expected conversions, and ROI using real dollar spending — quantifying the value of segment-specific campaign optimization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv('../data/preprocessed/customers_with_RFM_clustersts.csv')

# Segment label mapping (team standard)
SEGMENT_MAP = {0: 'Potential Loyals', 1: 'New / Occasional', 2: 'Loyal Customers', 3: 'At Risk/Low Value'}
if df['Segment'].dtype != 'object':
    df['Segment'] = df['Cluster'].map(SEGMENT_MAP)

print(f"Loaded {len(df)} customers across {df['Segment'].nunique()} segments")
df['Segment'].value_counts()

In [ ]:
CMP_COLS = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
SPENDING_COLS = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
CHANNEL_COLS = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
SEGMENT_ORDER = ['New / Occasional', 'At Risk/Low Value', 'Potential Loyals', 'Loyal Customers']

# Core aggregations — computed once, reused everywhere
seg_cmp_rates = df.groupby('Segment')[CMP_COLS].mean().reindex(SEGMENT_ORDER) * 100
seg_channel = df.groupby('Segment')[CHANNEL_COLS].mean().reindex(SEGMENT_ORDER)

seg_spend_pct = df.groupby('Segment')[SPENDING_COLS].sum().reindex(SEGMENT_ORDER)
seg_spend_pct = seg_spend_pct.div(seg_spend_pct.sum(axis=1), axis=0) * 100

seg_demo = df.groupby('Segment')[['Age', 'Income', 'Family_Size', 'Total_Spending', 'Total_Purchases']].mean().reindex(SEGMENT_ORDER)

df['Total_Accepted'] = df[CMP_COLS].sum(axis=1)

print("Core aggregations ready.")
seg_cmp_rates.round(1)

## 1. Which Campaigns Convert Which Segments?

Campaign acceptance rates (%) by segment — which campaigns resonate with which customer groups.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(seg_cmp_rates, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Acceptance Rate (%)'})
ax.set_title('Campaign Acceptance Rate (%) by Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Segment')
ax.set_xlabel('Campaign')
plt.tight_layout()
plt.show()

## 2. Channel Preferences by Segment

Average purchases per channel — shows whether each segment prefers web, catalog, or in-store.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
seg_channel.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.5)
ax.set_title('Average Purchases by Channel per Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Purchases')
ax.set_xlabel('Segment')
ax.legend(title='Channel', labels=['Web', 'Catalog', 'Store'])
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 3. Product Spend Profiles

Percentage of spend per product category by segment — reveals taste profiles and which products to promote.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('Set2', len(SPENDING_COLS))
seg_spend_pct.plot(kind='barh', stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Product Spend Distribution (%) by Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('% of Total Spend')
ax.set_ylabel('Segment')
ax.legend(title='Product', bbox_to_anchor=(1.02, 1), loc='upper left',
          labels=['Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold'])
plt.tight_layout()
plt.show()

## 4. Segment Overview

Four-panel dashboard: segment sizes, income distribution, overall response rate, and cross-tab of past campaign acceptance → Response.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Segment sizes
seg_sizes = df['Segment'].value_counts().reindex(SEGMENT_ORDER)
axes[0, 0].bar(range(len(seg_sizes)), seg_sizes.values, color=sns.color_palette('Set2', 4), edgecolor='black')
axes[0, 0].set_xticks(range(len(seg_sizes)))
axes[0, 0].set_xticklabels(SEGMENT_ORDER, rotation=15, ha='right', fontsize=9)
axes[0, 0].set_title('(a) Segment Sizes', fontweight='bold')
axes[0, 0].set_ylabel('# Customers')
for i, v in enumerate(seg_sizes.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# (b) Avg income by segment
incomes = seg_demo['Income']
axes[0, 1].bar(range(len(incomes)), incomes.values, color=sns.color_palette('Set2', 4), edgecolor='black')
axes[0, 1].set_xticks(range(len(incomes)))
axes[0, 1].set_xticklabels(SEGMENT_ORDER, rotation=15, ha='right', fontsize=9)
axes[0, 1].set_title('(b) Avg Income by Segment', fontweight='bold')
axes[0, 1].set_ylabel('Avg Income ($)')

# (c) Overall response rate by segment (% who accepted any campaign)
any_accepted = df.groupby('Segment')['Total_Accepted'].apply(lambda x: (x > 0).mean() * 100).reindex(SEGMENT_ORDER)
axes[1, 0].bar(range(len(any_accepted)), any_accepted.values, color=sns.color_palette('Set2', 4), edgecolor='black')
axes[1, 0].set_xticks(range(len(any_accepted)))
axes[1, 0].set_xticklabels(SEGMENT_ORDER, rotation=15, ha='right', fontsize=9)
axes[1, 0].set_title('(c) % Accepted Any Campaign', fontweight='bold')
axes[1, 0].set_ylabel('% of Segment')
for i, v in enumerate(any_accepted.values):
    axes[1, 0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# (d) Cross-tab heatmap: past campaign acceptance → Response
df['Accepted_Any_Past'] = df[['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']].max(axis=1)
cross = pd.crosstab(df['Accepted_Any_Past'], df['Response'], normalize='index') * 100
cross.index = ['No Past Acceptance', 'Past Acceptance']
cross.columns = ['No Response', 'Response']
sns.heatmap(cross, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1, 1], cbar_kws={'label': '%'})
axes[1, 1].set_title('(d) Past Acceptance → Response Rate', fontweight='bold')

plt.suptitle('Segment Overview Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Key Findings

- **Loyal Customers** and **Potential Loyals** dominate campaign acceptance — especially Cmp1, Cmp4, Cmp5, and Response. These are high-income, low-family-size segments.
- **New/Occasional** and **At Risk** barely respond to any campaign. Only Cmp3 and Response show marginal traction.
- **Channel split:** Loyal and Potential Loyal segments buy across all channels (especially catalog + store). Low-value segments are web-heavy with minimal catalog engagement.
- **Product profiles:** Wines and Meat dominate across all segments. Gold products skew slightly higher for New/Occasional and At Risk segments.
- **Past acceptance is the strongest predictor of future Response:** 40.7% response rate for past acceptors vs 8.3% for non-acceptors.

These patterns motivate a classifier that can predict campaign acceptance per segment and recommend optimal targeting strategy.

## 5. Campaign Response Classifier

Reshape data to long format (one row per customer × campaign), train a Gradient Boosting classifier to predict campaign acceptance, and extract feature importances.

In [ ]:
# Reshape to long format: each row = (customer, campaign, accepted)
id_vars = ['ID', 'Segment', 'Income', 'Age', 'Family_Size', 'Recency',
           'Total_Spending', 'Total_Purchases'] + CHANNEL_COLS

df_long = df.melt(id_vars=id_vars, value_vars=CMP_COLS,
                  var_name='Campaign', value_name='Accepted')

# One-hot encode Segment and Campaign
df_model = pd.get_dummies(df_long, columns=['Segment', 'Campaign'], drop_first=False)

# Feature columns (everything except ID and target)
feature_cols = [c for c in df_model.columns if c not in ['ID', 'Accepted']]

# GradientBoosting is tree-based (scale-invariant) — no scaling needed
X = df_model[feature_cols].values
y = df_model['Accepted'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

print(f"Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")
print(f"Target balance: {y.mean()*100:.1f}% positive (accepted)")
print(f"Features: {len(feature_cols)}")

In [ ]:
# Train Gradient Boosting Classifier
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                 random_state=42)
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Accepted']))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Accepted'], yticklabels=['Rejected', 'Accepted'])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importance
importances = pd.Series(gb.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.tail(15).plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Top 15 Feature Importances', fontweight='bold')
axes[1].set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Customer Acquisition Cost & Web Conversion Analysis

With campaign acceptance rates quantified per segment, we can now model the **cost of acquiring a customer** under different targeting strategies.

**Assumptions:**
- $500 campaign budget, $2 per web visit → 250 visits per campaign
- All campaigns (Cmp1–5) treated as web campaigns (consistent delivery channel)
- **Web Conversion Rate** = `NumWebPurchases / (NumWebVisitsMonth × tenure_months)`

**Three targeting strategies compared:**
| # | Strategy | Budget Split | Audience | Acceptance Rate |
|---|----------|-------------|----------|-----------------|
| S1 | All campaigns × Everyone | $100/cmp × 5 | Whole segment | Avg across Cmp1–5 |
| S2 | Best campaign × Everyone | $500 on 1 cmp | Whole segment | Best cmp rate |
| S3 | Best campaign × Responders only | $500 on 1 cmp | Past responders | Best cmp rate among responders |

In [ ]:
# === CAC & Web Conversion Computation ===

BUDGET = 500
COST_PER_VISIT = 2
VISITS = BUDGET / COST_PER_VISIT  # 250

# Campaign columns excluding Response (Response is the target, not a campaign we "run")
CMP_ONLY = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']

# Web conversion rate per customer
df['Customer_Tenure_Months'] = df['Customer_Tenure_Days'] / 30
df['Web_Conversion_Rate'] = np.where(
    (df['NumWebVisitsMonth'] > 0) & (df['Customer_Tenure_Months'] > 0),
    df['NumWebPurchases'] / (df['NumWebVisitsMonth'] * df['Customer_Tenure_Months']),
    0
)

cac_rows = []

for segment in SEGMENT_ORDER:
    seg_data = df[df['Segment'] == segment]
    seg_responders = seg_data[seg_data['Total_Accepted'] > 0]

    # Campaign acceptance rates for this segment (excluding Response)
    cmp_rates = seg_data[CMP_ONLY].mean()
    best_cmp_name = cmp_rates.idxmax()
    best_cmp_rate = cmp_rates.max()

    # S1: spray-and-pray — average acceptance across all 5 campaigns
    s1_rate = cmp_rates.mean()

    # S2: best campaign × everyone
    s2_rate = best_cmp_rate

    # S3: best campaign × past responders only
    if len(seg_responders) > 0:
        s3_rate = seg_responders[CMP_ONLY].mean().max()
    else:
        s3_rate = 0

    # CAC = budget / (visits × acceptance_rate)
    s1_cac = BUDGET / (VISITS * s1_rate) if s1_rate > 0 else np.inf
    s2_cac = BUDGET / (VISITS * s2_rate) if s2_rate > 0 else np.inf
    s3_cac = BUDGET / (VISITS * s3_rate) if s3_rate > 0 else np.inf

    # Expected conversions per $500
    s1_conv = VISITS * s1_rate
    s2_conv = VISITS * s2_rate
    s3_conv = VISITS * s3_rate

    # Web conversion rate
    wcr = seg_data['Web_Conversion_Rate'].mean()

    cac_rows.append({
        'Segment': segment,
        'Best_Campaign': best_cmp_name,
        'S1_Rate': s1_rate,
        'S2_Rate': s2_rate,
        'S3_Rate': s3_rate,
        'S1_CAC': s1_cac,
        'S2_CAC': s2_cac,
        'S3_CAC': s3_cac,
        'S1_Conversions': s1_conv,
        'S2_Conversions': s2_conv,
        'S3_Conversions': s3_conv,
        'Web_Conversion_Rate': wcr
    })

cac_df = pd.DataFrame(cac_rows).set_index('Segment')

print("=== CAC Analysis ($500 budget, $2/visit) ===\n")
display_cols = ['Best_Campaign', 'S1_CAC', 'S2_CAC', 'S3_CAC', 'S1_Conversions', 'S2_Conversions', 'S3_Conversions', 'Web_Conversion_Rate']
cac_df[display_cols].round(2)

In [ ]:
# === CAC Visualization ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(SEGMENT_ORDER))
width = 0.25

# Left: CAC per segment by strategy
cac_vals = cac_df[['S1_CAC', 'S2_CAC', 'S3_CAC']].replace(np.inf, np.nan)
bars1 = axes[0].bar(x - width, cac_vals['S1_CAC'], width, label='S1: Spray & Pray', color='#e74c3c', edgecolor='black')
bars2 = axes[0].bar(x, cac_vals['S2_CAC'], width, label='S2: Best Cmp × All', color='#f39c12', edgecolor='black')
bars3 = axes[0].bar(x + width, cac_vals['S3_CAC'], width, label='S3: Best Cmp × Responders', color='#27ae60', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(SEGMENT_ORDER, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('Customer Acquisition Cost ($)')
axes[0].set_title('CAC by Targeting Strategy', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)

# Right: Expected conversions per $500
bars1 = axes[1].bar(x - width, cac_df['S1_Conversions'], width, label='S1: Spray & Pray', color='#e74c3c', edgecolor='black')
bars2 = axes[1].bar(x, cac_df['S2_Conversions'], width, label='S2: Best Cmp × All', color='#f39c12', edgecolor='black')
bars3 = axes[1].bar(x + width, cac_df['S3_Conversions'], width, label='S3: Best Cmp × Responders', color='#27ae60', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(SEGMENT_ORDER, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('Expected Conversions per $500')
axes[1].set_title('Conversions by Targeting Strategy', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Customer Acquisition Cost & Conversion Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Per-segment summary
print("\n=== Per-Segment Summary ===")
for seg in SEGMENT_ORDER:
    row = cac_df.loc[seg]
    if row['S1_CAC'] != np.inf and row['S3_CAC'] != np.inf:
        reduction = (1 - row['S3_CAC'] / row['S1_CAC']) * 100
        print(f"  {seg}: CAC drops {reduction:.0f}% with targeted approach "
              f"(${row['S1_CAC']:.2f} → ${row['S3_CAC']:.2f}) | "
              f"Web Conversion Rate: {row['Web_Conversion_Rate']:.3f}")
    else:
        print(f"  {seg}: Insufficient acceptance data for CAC comparison | "
              f"Web Conversion Rate: {row['Web_Conversion_Rate']:.3f}")

In [ ]:
# === ROI Analysis ===
# Pull real dollar spending from raw data (preprocessed values are log-transformed)
raw = pd.read_csv('../data/raw/marketing_campaign.csv', sep='\t')
RAW_SPEND_COLS = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
raw['Real_Total_Spending'] = raw[RAW_SPEND_COLS].sum(axis=1)
df = df.merge(raw[['ID', 'Real_Total_Spending']], on='ID', how='left')

avg_real_spend = df.groupby('Segment')['Real_Total_Spending'].mean().reindex(SEGMENT_ORDER)

roi_rows = []
for seg in SEGMENT_ORDER:
    row = cac_df.loc[seg]
    spend = avg_real_spend[seg]

    # Revenue = conversions × avg real spend per customer
    s1_rev = row['S1_Conversions'] * spend
    s2_rev = row['S2_Conversions'] * spend
    s3_rev = row['S3_Conversions'] * spend

    # ROI = (revenue - cost) / cost × 100
    s1_roi = ((s1_rev - BUDGET) / BUDGET) * 100
    s2_roi = ((s2_rev - BUDGET) / BUDGET) * 100
    s3_roi = ((s3_rev - BUDGET) / BUDGET) * 100

    # Incremental: S3 vs S1
    incr_rev = s3_rev - s1_rev
    incr_customers = row['S3_Conversions'] - row['S1_Conversions']

    roi_rows.append({
        'Segment': seg,
        'Avg_Spend_Real': round(spend, 2),
        'S1_ROI_%': round(s1_roi, 1),
        'S2_ROI_%': round(s2_roi, 1),
        'S3_ROI_%': round(s3_roi, 1),
        'S1_Revenue': round(s1_rev, 2),
        'S3_Revenue': round(s3_rev, 2),
        'Incremental_Revenue': round(incr_rev, 2),
        'Incremental_Customers': round(incr_customers, 1)
    })

roi_df = pd.DataFrame(roi_rows).set_index('Segment')

# ROI bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(SEGMENT_ORDER))
width = 0.25

ax.bar(x - width, roi_df['S1_ROI_%'], width, label='S1: Spray & Pray', color='#e74c3c', edgecolor='black')
ax.bar(x, roi_df['S2_ROI_%'], width, label='S2: Best Cmp × All', color='#f39c12', edgecolor='black')
ax.bar(x + width, roi_df['S3_ROI_%'], width, label='S3: Best Cmp × Responders', color='#27ae60', edgecolor='black')
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(SEGMENT_ORDER, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('ROI (%)')
ax.set_title('Campaign ROI by Targeting Strategy ($500 Budget)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Summary table
print("\n=== ROI Summary ===")
print(roi_df[['Avg_Spend_Real', 'S1_ROI_%', 'S2_ROI_%', 'S3_ROI_%', 'Incremental_Revenue', 'Incremental_Customers']])

# Per-segment narrative
print("\n=== Targeting Impact ===")
for seg in SEGMENT_ORDER:
    row = roi_df.loc[seg]
    cac_row = cac_df.loc[seg]
    print(f"  {seg}: By targeting {cac_row['Best_Campaign']} to previous responders, "
          f"we gain {row['Incremental_Customers']:.1f} additional customers worth "
          f"${row['Incremental_Revenue']:,.0f} in expected revenue — "
          f"ROI improves from {row['S1_ROI_%']:.0f}% to {row['S3_ROI_%']:.0f}%")

## Conclusions

**Customer Acquisition Cost (CAC) — targeted vs spray-and-pray:**
- Targeted campaign selection (S3) dramatically reduces CAC across all viable segments compared to distributing budget equally across all campaigns (S1).
- The ROI improvement from S1 → S3 quantifies the value of segment-specific campaign optimization using the same $500 budget.

**Campaign response patterns:**
- Income, Recency, and Total Spending are the strongest predictors of campaign acceptance — confirming that RFM segmentation aligns with campaign responsiveness.
- **Past campaign acceptance is the single best signal for future response:** 40.7% response rate for past acceptors vs 8.3% for non-acceptors.
- Acceptance rates vary significantly by segment and campaign number, but the data contains no metadata about campaign delivery channel or type — differences reflect audience response, not channel effects.

**Web conversion context:**
- Web conversion rates provide baseline context for how efficiently each segment converts visits into purchases, independent of campaign targeting.

**Connection to RFM clustering growth simulation:**
- The CAC and ROI analysis here complements the $500-budget growth simulation in the RFM Clustering notebook, providing the campaign-level tactical view alongside the segment-level strategic view.

**Next steps:**
- Validate targeting recommendations with A/B testing on next campaign cycle
- Explore time-series patterns (does Recency predict which *specific* campaign converts?)
- Feed segment × campaign predictions into automated targeting pipeline